# Step 2: Filter notebook

In [ ]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
import rasterio
import numpy as np
import os
import folium
import json
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from feature_to_network import *

## Imports

#### Import des segments

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-2'

    attribute_source_path = 'source_path_GG'
    file_name = 'file_name_GG'  # column name in attributs_info excel

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-2'

    attribute_source_path = 'source_path_GE'
    file_name = 'file_name_GE'  # column name in attributs_info excel

save_filtered_attributes = True


# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


In [ ]:
import geopandas as gpd
from shapely.geometry.base import BaseGeometry

# Clean geometries safely
def clean_any_geom(geom):
    if geom is None:
        return None
    if not isinstance(geom, BaseGeometry):
        return None
    if geom.is_valid:
        return geom
    try:
        fixed = geom.buffer(0)
        if fixed is None or fixed.is_empty:
            return geom
        return fixed
    except Exception:
        return geom
        




# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")


#### Import des attributs 

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

In [ ]:
attributs_info

In [ ]:
import fiona
from pathlib import Path

gpkg_path = Path(input_file_path) / "attributs" / territory / "osm" / "osm_attributes.gpkg"
print("gpkg_path:", gpkg_path)
print("exists:", gpkg_path.exists())

print(fiona.listlayers(str(gpkg_path)))



**Attributs OSM**

**Attribut stationnement velo**


Amélioration: donner un meilleur score selon la capacité, présence abris ou non

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_velo'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

# --- Capacity cleaning (handles empty strings, None, non-numeric) ---
gdf["capacity_num"] = pd.to_numeric(
    gdf["capacity"].astype(str).str.strip().replace({"": None, "nan": None, "None": None}),
    errors="coerce"
)

# --- Score using capacity, fallback to 1 ---
gdf["score"] = gdf["capacity_num"].fillna(1)

# (optional) avoid zero/negative capacities becoming weird scores
gdf.loc[gdf["score"] <= 0, "score"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut location**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'location'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut zone apaisée**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Atttribut Vitesse motorisée**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse_motorisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.speed_kph>25, 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Contre-sens cyclable**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'sens_inverse'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_AMEN=="Contresens", 'filtered'] = 1
gdf.loc[gdf.TYPE_AMEN=="Dérogation 2R", 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Giratoire**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'giratoire'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut revêtement**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'revetement'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
surface_score = {
    "asphalt": 1,
    "concrete": 1,
    "concrete:plates": 1,
    "concrete:lanes": 1,
    "paved": 0,
    "paving_stones": 0,
    "sett": 0,
    "metal": 0,
    "wood": 0,

    "fine_gravel": 0,
    "compacted": 0,
    "gravel": 0,
    "pebblestone": 0,

    "ground": 0,
    "dirt": 0,
    "earth": 0,
    "grass": 0,
    "grass_paver": 0,
    "sand": 0,
    "rock": 0,
    "unpaved": 0,
    "woodchips": 0,
}


gdf["surface_score"] = gdf["surface"].map(surface_score)
gdf.loc[gdf["surface_score"] == 1, "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Aménité**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")
# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
rez_actif_set = {
"restaurant","cafe","bar","pub","fast_food","ice_cream","biergarten",
"bank","atm","bureau_de_change",
"cinema","theatre","nightclub","casino",
"marketplace","vending_machine"
}

gdf.loc[gdf["amenity"].isin(rez_actif_set), "filtered"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


**Attribut borne réparation**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'borne_reparation'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut éclairage**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eclairage'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.lit=='yes', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Confort thermique**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Sample every 10th point
    rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Lac et cours d'eau**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eau'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


**Attribut Canopée**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'canopee'

###--------------------

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut pollution air**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'air'

###--------------------

    
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Sample every 10th point
    rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'no_2': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['no_2'].describe())

# Save
print(f"Saving {attribute}: can take up to 3mn")
save(save_filtered_attributes, row, gdf, attribute)

**Attribut piste**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'piste'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut bande**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bande'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut environnement agréable (alentours)**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'alentours'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
landuse_agreable = {
    # 3 = très agréable
    "forest",
    "grass",
    "meadow",
    "recreation_ground"
}

gdf.loc[gdf["landuse"].isin(landuse_agreable), "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut conflits usage**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'conflit_md'

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
gdf['severity_score'] = 0
gdf['severity_score'] = gdf['severity'].map({'high' : 3, 'medium':2, 'low':1})

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut service vélo**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'service_velo'

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut parking abris**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'parking_abris'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf["covered"] == "yes", "filtered"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut accident**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf["AccidentInvolvingBicycle"] == "true", "filtered"] = 1

gdf['severity_score'] = 0
severity_mapping = {
    "accident avec blessés légers":1,
    "accident avec blessés graves":2,
    "accident avec tués":3
}
gdf['severity_score'] = gdf['AccidentSeverityCategory_fr'].map(severity_mapping)

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)